In [4]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages
from langgraph.graph import MessagesState
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

import datetime

load_dotenv()

True

In [5]:
# 想讓 LLM 調用的 function / tool

def get_weather(location: str, date: str = None) -> str:
    if not date:
        date = (datetime.date.today() + datetime.timedelta(days=1)).isoformat()
    return f"{location} 在 {date} 的天氣是晴時多雲，氣溫約 26~32 °C。"

In [6]:
llm = init_chat_model("openai:gpt-4.1-mini")

### 工具揭露

In [7]:
tool_expose_prompt = """｀
你可以使用 get_weather 函數來查詢天氣，它需要以下參數:
- location (必填): 要查詢的地點名稱
- date (選填): 要查詢的日期，格式為 YYYY-MM-DD。如果不提供，預設會查詢明天的天氣。

範例:
get_weather(location="台北", date="2024-01-01")
get_weather(location="台中")

如果要使用工具，勿必要回傳上述語法，不要省略任何字元，也不要自行增長任何字元，否則會導致工具無法正常使用
"""

### 使用者對話

In [11]:
user_input = "請問明天台北天氣如何"

messages = [
    SystemMessage(content=tool_expose_prompt),
    HumanMessage(content=user_input),
]

response = llm.invoke(messages)
print(response.content)

get_weather(location="台北")


### 解析 LLM 輸出，判斷是否使用 function

In [12]:
if "get_weather" in response.content:
    # 從 response.content 進行解析
    eval(response.content)
else:
    print("沒有使用工具")
    print(response.content)
